# Fehlende Zugangsbeschränkungen aus Mapillary-Verbotszeichen → MapRoulette

Findet Verbotszeichen (DE:250, 251, 253, 254, 255, 259, 260), in deren Nähe kein OSM-Weg ein
Tag für den gesperrten Verkehr trägt — etwa ein „Verbot für Radfahrer“ ohne `bicycle=*`,
`vehicle=*` oder `access=*` im Umkreis von 30 m. Daraus werden Aufgaben für die
MapRoulette-Challenge 54768. Läuft von Hand, wenn eine neue Runde fällig ist.

Aufgebaut wie die `maproulette_tasks.ipynb` der übrigen Kampagnen. Die Schritte liegen in
[`../campaign.py`](../campaign.py); was diese Kampagne ausmacht — Zeichen, welche Tags je
Zeichen zählen, Aufgabentext — in [`../ar_campaign.py`](../ar_campaign.py).

**Ablauf**

| Schritt | Funktion |
| --- | --- |
| Verkehrszeichen von data.vizsim.de spiegeln | `campaign.sync_features` |
| Verbotszeichen einlesen | `campaign.load_features` |
| auf Deutschland beschneiden | `campaign.clip_to_boundary` |
| frische / kurzlebige Schilder aussortieren | `campaign.filter_stable_signs` |
| Abstand zum nächsten Weg mit passendem Tag | `ar.distance_to_matching_way` |
| bereits bestehende Aufgaben abziehen | `campaign.drop_near_existing_tasks`, `campaign.drop_known_task_names` |
| **neuestes** Mapillary-Bild je Zeichen holen | `campaign.newest_image_ids` |
| GeoJSON für MapRoulette schreiben | `campaign.build_maproulette_geojson` mit `ar.aufgabe` |
| in die Challenge hochladen (nur mit `CAMPAIGN_UPLOAD=1`) | `campaign.upload_tasks` |

Die Challenge holt ihre Aufgaben aus `maproulette_tasks_missing-access-restrictions.geojson` auf
GitHub: Notebook laufen lassen, Datei committen und pushen, in MapRoulette „Rebuild tasks“.
Abschnitt 9 lädt stattdessen direkt über die API hoch — sobald MapRoulette Skripten wieder
einen Zugang gibt (siehe [README](README.md)).

## Einstellungen

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# campaign.py and ar_campaign.py live one level up. ".." works because Jupyter and
# nbconvert run the kernel in the notebook's directory - the same assumption as
# ../../output/ and ../utils/.
sys.path.insert(0, "..")
import ar_campaign as ar
import campaign

# Must match the date in ../0_prepare_osm_network.ipynb, which writes the access extract.
set_date = "261017"

ordner_zeichen = Path("../../output")
pfad_wege = Path(f"../utils/processed_osm_files/processed_access_restrictions_germany_{set_date}.parquet")
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")

# Only signs last seen after this date ...
zuletzt_gesehen_nach = "2025-07-01"
# ... and seen over at least this many months - filters out temporary signs.
mindest_standzeit_monate = 9

# Whether only roads and paths can answer a sign. False, as in the 2026 round: a
# parking lot with access=private nearby counts as mapped, too. True gave 1,054
# candidates more on the data of 17.10.2026 - see ar.read_access_ways.
nur_strassen = False

challenge_id = 54768

# MapRoulette pulls the tasks from this file on GitHub (remoteGeoJson), so it keeps
# its name. Once uploads by script work again, section 9 replaces commit, push and
# "Rebuild tasks".
ziel_geojson = Path("maproulette_tasks_missing-access-restrictions.geojson")

# From the environment, so a run on the server needs no edit to this notebook:
#   CAMPAIGN_SYNC=0    read ../../output/ as it is. On the server it is what the
#                      weekly run produced, and mirroring would overwrite its manifest.
#   CAMPAIGN_UPLOAD=1  upload the tasks to the challenge - refused by MapRoulette
#                      until it has tokens for scripts, see the README. Without it,
#                      section 9 only says how many it would upload.
sync = os.environ.get("CAMPAIGN_SYNC", "1") != "0"
hochladen = os.environ.get("CAMPAIGN_UPLOAD") == "1"

spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

print("Zeichen:", ", ".join(f"DE:{nummer}" for nummer, _ in ar.ZEICHEN.values()))
print(f"Suchradius {ar.SUCHRADIUS_M:.0f} m")
print(f"spiegeln: {sync}, hochladen: {hochladen}")

## 1 · Verkehrszeichen holen und einlesen

In [ ]:
if sync:
    # Fetches only what is missing locally or newer on data.vizsim.de.
    campaign.sync_features(ordner_zeichen)

manifest = json.loads((ordner_zeichen / "ml-ts_metadata.json").read_text(encoding="utf-8"))
print("Datenstand:", manifest["processed_date"])
print("im letzten Lauf unvollständig:", manifest.get("last_run_incomplete") or "nichts")

In [ ]:
# Expected number of states from the tile cache; None without one.
erwartet = campaign.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

signs = campaign.load_features(
    ordner_zeichen,
    prefix=campaign.PREFIX_ZEICHEN,
    values=ar.ZEICHEN,
    columns=spalten,
    expect_files=erwartet,
)
# The parquets are cut along zoom-14 tiles, and border tiles reach abroad - where
# the OSM extract has no ways, so every sign there would become a task.
signs = campaign.clip_to_boundary(signs, campaign.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5))
signs["last_seen_at"].str[:7].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Zeichen")

signs = campaign.filter_stable_signs(signs, zuletzt_gesehen_nach, mindest_standzeit_monate)

signs["last_seen_at"].str[:7].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1], title=f"zuletzt gesehen nach {zuletzt_gesehen_nach}, ≥ {mindest_standzeit_monate} Monate"
)
plt.tight_layout()

## 3 · Abstand zum nächsten Weg mit passendem Tag

Welche Tags zählen, hängt vom Zeichen ab (`ar.SCHLUESSEL`): für „Verbot für Radfahrer“
`bicycle=*`, `vehicle=*` oder `access=*`, für „Verbot für Fußgänger“ `foot=*` oder `access=*`
und so weiter. Der Wert spielt keine Rolle, wie in der Runde 2026.

Wie bisher zählen auch Flächen, vor allem Parkplätze mit `access=private`. Mit
`nur_strassen = True` zählen nur Straßen und Wege (`highway=*`); auf den Daten vom 17.10.2026
gab das 1.054 Kandidaten mehr.

In [ ]:
wege = ar.read_access_ways(pfad_wege, nur_strassen=nur_strassen)

signs["dist_access_m"] = ar.distance_to_matching_way(signs, wege)
del wege

ohne = signs["dist_access_m"] == float("inf")
print(f"ohne passendes Tag in {ar.SUCHRADIUS_M:.0f} m: {ohne.sum()} von {len(signs)}")

## 4 · Kandidaten

In [ ]:
kandidaten = signs[signs["dist_access_m"] == float("inf")].reset_index(drop=True)

print(f"Kandidaten: {len(kandidaten)}")
print(kandidaten["value"].map(lambda value: f"DE:{ar.ZEICHEN[value][0]}").value_counts().to_string())
kandidaten.plot(figsize=(6, 8), markersize=2)

## 5 · Bereits bestehende Aufgaben abziehen

Alles außer Fixed, Created und Skipped blockiert eine Neuanlage in 30 m. Außerdem fällt jedes
Zeichen weg, dessen ID Name einer erledigten Aufgabe ist: MapRoulette überschriebe die alte
Aufgabe, statt eine neue anzulegen, und sie bliebe erledigt. Ist die Aufgabe dagegen noch
offen — Created oder Skipped —, bleibt das Zeichen drin, und der Rebuild aktualisiert sie.

In [ ]:
api_key = campaign.load_token(pfad_config, "API_KEY_MAPROULETTE")

# Everything but Fixed / Created / Skipped blocks a new task nearby: what was ticked
# off as "false positive" or "already fixed" must not come back.
bestehende = campaign.load_challenge_tasks(challenge_id, api_key)
print(f"blockierende Aufgaben in Challenge {challenge_id}: {len(bestehende)}")
kandidaten = campaign.drop_near_existing_tasks(kandidaten, bestehende, radius_m=ar.SUCHRADIUS_M)

# A sign whose id names a closed task would only update that task - it would stay
# closed. Open tasks of the same name stay in: the rebuild refreshes them.
kandidaten = campaign.drop_known_task_names(kandidaten, campaign.challenge_task_names(challenge_id, api_key))
print(f"neu anzulegen: {len(kandidaten)}")

## 6 · Neuestes Mapillary-Bild

Die Runde 2026 nahm `images[-1]` — das letzte Element einer Liste, die nicht nach
Aufnahmezeit sortiert ist. Hier kommt `captured_at` über die Graph-API gleich mit
(`fields=id,images{id,captured_at}`), und bis zu 50 Zeichen gehen in eine Anfrage.

In [ ]:
bilder = campaign.newest_image_ids(kandidaten["id"], campaign.load_token(pfad_config))

# Join on the string index: Mapillary ids exceed 2**53 and would be rounded as numbers.
kandidaten = kandidaten.set_index(kandidaten["id"].astype(str)).join(bilder).reset_index(drop=True)

kandidaten[["id", "value", "last_seen_at", "image_id", "image_captured_at"]].head()

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der das Zeichen erkannt wurde.
Stimmt die Bildauswahl, ist das verlinkte Bild ungefähr von diesem Tag.

In [ ]:
abstand_tage = (
    kandidaten["image_captured_at"].dt.tz_localize(None) - pd.to_datetime(kandidaten["last_seen_at"])
).dt.days.abs()

print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

if abstand_tage.notna().any():
    abstand_tage.plot(kind="hist", bins=40, figsize=(8, 3), title="Tage zwischen verlinktem Bild und last_seen_at")

## 7 · GeoJSON für MapRoulette schreiben

Die Datei behält ihren Namen, weil MapRoulette sie von GitHub holt. Ihr bisheriger Inhalt wird
**vor** dem Überschreiben eingelesen — sonst gäbe es im nächsten Schritt nichts zu vergleichen.

In [ ]:
vorheriger_stand = campaign.read_geojson(ziel_geojson)

aufgaben = campaign.build_maproulette_geojson(kandidaten, ar.aufgabe)
campaign.write_geojson(aufgaben, ziel_geojson)

if aufgaben["features"]:
    print(aufgaben["features"][0]["properties"]["instruction"])

## 8 · Vergleich mit dem vorherigen Stand

Stellt die Aufgaben-IDs des bisherigen Datei-Inhalts denen dieses Laufs gegenüber. **anderes
Bild** zählt, wie viele Aufgaben vorher ein anderes Foto verlinkten.

In [ ]:
if vorheriger_stand:
    campaign.print_comparison(campaign.compare_task_sets(vorheriger_stand, aufgaben))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

## 9 · In die Challenge hochladen

Direkt über die MapRoulette-API (`addFileTasks`), ohne Commit und ohne „Rebuild tasks“. Die
Aufgaben, die schon in der Challenge stehen, bleiben, wie sie sind. Danach prüft
`upload_tasks`, ob jede Aufgabe angekommen ist — MapRoulette meldet Erfolg auch dann, wenn
der Import intern scheitert.

Läuft nur mit `CAMPAIGN_UPLOAD=1`, und **noch gar nicht**: Seit maproulette-backend v4.11.0
(Oktober 2026) lehnt MapRoulette API-Keys ab; der angekündigte Ersatz, Personal Access Tokens,
fehlt noch. Bis dahin: Datei committen, pushen, „Rebuild tasks“.

In [ ]:
if hochladen:
    nachher = campaign.upload_tasks(
        challenge_id, aufgaben, api_key, data_origin_date=manifest["processed_date"][:10]
    )
    print(pd.Series(nachher).value_counts().to_string())
else:
    print(f"nicht hochgeladen ({len(aufgaben['features'])} Aufgaben) - dafür CAMPAIGN_UPLOAD=1 setzen")

---

## Challenge-Beschreibung (für MapRoulette)

## Add missing access restrictions based on Mapillary signs (Germany)

This campaign highlights regulatory prohibition signs from Mapillary that currently do not
appear to have a nearby matching access-related tag in OSM.

Focus in V1 is on signs 250, 251, 253, 254, 255, 259 and 260.

1. Öffne den Ort in **Mapillary**, in einer Kartenansicht und in deinem OSM-Editor.
2. Prüfe, ob das erkannte Verkehrszeichen tatsächlich für den betroffenen Weg gilt.
3. Wenn in OSM noch kein passendes access-bezogenes Tag vorhanden ist, trage das fehlende Tag ein.
4. Achte auf Zusatzzeichen, Ausnahmen wie `Anlieger frei`, zeitliche Einschränkungen und Fahrtrichtung.
5. Wenn die Situation unklar ist oder das Schild nicht eindeutig zum Weg gehört, löse die Aufgabe nicht blind.

---

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```